# Exploracion con NLP y LLMs

ideas generales

- Cómo se presenta y se nombra a las personas que aparecen en el vídeo: nombre, título, cargo, profesión (cruzado con el OCR). ¿Se nombra con título y cargo por igual a mujeres y hombres? ("la Doctora María" vs. "mi compañera María")
	- Task 1: LLM. Extraer cada referencia a una persona presente: cómo se la nombra, nombre, título/cargo y género.

- Temas según el género de quien habla: estereotípicos (cuidados, familia, vida sentimental, ser mujer) vs. profesionales (tecnología, finanzas, liderazgo, su trabajo).
	- Task 2: Zero-shot o LLM con una lista fija de temas, aplicado a cada turno. Con el género del hablante (del audio), sale el reparto de temas por género.
    - Aquí a lo mejor tengo que linkear a qién se refiere el topic, no solo quién habla de él

- Referentes: ¿se citan personas destacadas femeninas o solo masculinas?
	- Task 3: NER con GLiNER para encontrar los nombres y buscarlos en Wikidata para saber su género.

- Rol de cada persona en la pieza. Primero el formato (entrevista, reportaje con voz en off, narración, anuncio, debate…) y después el rol de cada hablante según el formato: experta/o, entrevistador/a, presentador/a, narrador/a en off, testimonio, personaje… ¿Quién tiene los roles de autoridad?
	- Task 4: LLM con la transcripción completa: formato de la pieza y rol de cada hablante.

- Desprecios, desmerecimiento, micromachismos o faltas de respeto, y su dirección (mujer → hombre, hombre → mujer). Las interrupciones se miden en el pipeline de audio.
	- Task 5: LLM con criterios escritos y revisión humana o probar modelos zero shot entrenados para esta tarea.

- Contribución al contenido: si quitamos todas las intervenciones femeninas, ¿sigue teniendo sentido la pieza?
	- Task 6: LLM resume la pieza completa: % de las ideas del resumen que vienen de intervenciones de mujeres. Complemento: LLM como juez con una rúbrica de escala cerrada.


- Task 7: Cuantificar con word2vec traduciendo al inglés, medir con sesgo de estereotipo
- Validación: anotar a mano 2–3 vídeos para comprobar cada task.

## 0. Configuración

Entrada: la transcripción del pipeline de audio (CSV del contrato: `tiempo_i`, `tiempo_f`, `transcripcion`, `genero_inferido`).

In [1]:
import os
from typing import Literal
import pandas as pd
import litellm
from pydantic import BaseModel, Field

TRANSCRIPT = "clip_quanta.csv"            # <- CSV de la transcripción

BASE = os.environ.get("OVH_BASE_URL", "https://oai.endpoints.kepler.ai.cloud.ovh.net/v1")
TOKEN = "".join((os.environ.get("OVH_AI_ENDPOINTS_ACCESS_TOKEN") or os.environ.get("OVH_API_TOKEN") or "").split())
MODEL = "openai/Meta-Llama-3_3-70B-Instruct"   # "openai/" = endpoint compatible con OpenAI (OVH)
assert TOKEN, "Falta el token: define la variable de entorno OVH_AI_ENDPOINTS_ACCESS_TOKEN"

turnos = pd.read_csv(TRANSCRIPT)
# La transcripción en texto: un turno por línea, con su minuto y segundo
texto = "\n".join(f"[{int(t // 60):02d}:{int(t % 60):02d}] {txt}" for t, txt in zip(turnos.tiempo_i, turnos.transcripcion))
print(texto[:500])

[00:00] declarada sobre aquesta qüestió. Ella és la Carina Givert, és la directora i cofundadora de l'IDEA i la coneixeu, Carina. Benvinguda. Hola. Et puc fer la presentació oficial? Sí, és encantada. Presentació oficial.
[00:15] Carina Givert és catedràtica a la Universitat Politécnica de Catalunya Barcelona Tech. Llicenciada i doctora en informàtica amb especialitzacions en estadística computacional i intel·ligència artificial. directora i cofundadora de l'IDEAI Centre de Recerca de la UPC en 


## Task 1. Cómo se nombra a las personas de la conversación

Cada alusión a una persona que participa en la pieza (presentaciones, cuando alguien se dirige a ella, referencias): texto literal, nombre, título o cargo y género. Salida estructurada con Pydantic; el prompt incluye ejemplos (*few-shot*).

In [2]:
class Alusion(BaseModel):
    texto_literal: str = Field(description="Fragmento literal de la transcripción donde se alude a la persona")
    nombre: str | None = Field(description="Nombre propio de la persona; null si no se dice")
    titulo_cargo: str | None = Field(description="Título, cargo o profesión; null si no se dice")
    genero: Literal["F", "M"] | None = Field(description="Género gramatical con el que se alude a la persona; null si no se sabe, o no es binario.")

class Alusiones(BaseModel):
    alusiones: list[Alusion]

In [ ]:
PROMPT_T1 = """Analizas la transcripción automática de un vídeo (puede estar en español, catalán o inglés y tener errores de reconocimiento de voz).
Extrae TODAS las alusiones a personas que participan en la conversación: cuando se las presenta, cuando alguien se dirige a ellas o cuando se habla de ellas.
No incluyas a personas que solo se citan y no participan explícitamente (personajes históricos, famosos, ellos, ellas, nosotras...).

Para cada alusión devuelve:
- texto_literal: el fragmento exacto de la transcripción.
- nombre: el nombre propio tal como aparece; null si no se dice.
- titulo_cargo: título, cargo o profesión (doctora, catedrática de la UPC, director del hospital...); null si no se dice.
  Las relaciones NO son cargos: "mi compañera", "mi amigo", "la mujer de..." -> null.
- genero: "F" o "M" según cómo se alude a la persona (artículos, pronombres, concordancia); null si no se sabe.

Ejemplos:
Texto: "Hoy nos acompaña la doctora Marta Ruiz, investigadora del CSIC."
-> {"texto_literal": "la doctora Marta Ruiz, investigadora del CSIC", "nombre": "Marta Ruiz", "titulo_cargo": "doctora, investigadora del CSIC", "genero": "F"}
Texto: "Gracias, Luis, por estar aquí."
-> {"texto_literal": "Gracias, Luis", "nombre": "Luis", "titulo_cargo": null, "genero": "M"}
Texto: "Mi compañera nos lo explica ahora."
-> {"texto_literal": "Mi compañera", "nombre": null, "titulo_cargo": null, "genero": "F"}
Texto: "Pere Vila, director del hospital, ha atendido a los medios."
-> {"texto_literal": "Pere Vila, director del hospital", "nombre": "Pere Vila", "titulo_cargo": "director del hospital", "genero": "M"}"""

resp = litellm.completion(
    model=MODEL, api_base=BASE, api_key=TOKEN, temperature=0,
    messages=[{"role": "system", "content": PROMPT_T1},
              {"role": "user", "content": texto}],
    response_format=Alusiones,             
)
alusiones = Alusiones.model_validate_json(resp.choices[0].message.content)
t1 = pd.DataFrame([a.model_dump() for a in alusiones.alusiones])
t1